# Ex 8: Association Rules and PCA

**Name:** Midhun P  
**Roll Number:** 24BAD069

In [ ]:
print("Midhun P 24BAD069")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
import warnings

warnings.filterwarnings('ignore')

---

## SCENARIO 1 – ASSOCIATION RULE MINING USING APRIORI
**Problem Statement:** Identify frequent itemsets and generate association rules from transactional data using the Apriori algorithm.

### 1. Load Transactional Dataset

In [ ]:
try:
    df_groceries = pd.read_csv('Groceries_dataset.csv')
except Exception:
    df_groceries = pd.read_csv('c:/Users/Asus/OneDrive - Kumaraguru College of Technology/Desktop/Sem4/ML/lab/ex_8/Groceries_dataset.csv')

print(df_groceries.head())

### 2. Data Preprocessing (One-Hot Encoding)

In [ ]:
# Group by Member_number and Date to form transactions
if 'Member_number' in df_groceries.columns and 'Date' in df_groceries.columns:
    df_groceries['Transaction'] = df_groceries['Member_number'].astype(str) + "_" + df_groceries['Date'].astype(str)
    item_col = 'itemDescription'
else:
    # Fallback structure
    df_groceries['Transaction'] = df_groceries.index
    item_col = df_groceries.columns[-1]

basket = (df_groceries.groupby(['Transaction', item_col])[item_col]
          .count().unstack().reset_index().fillna(0).set_index('Transaction'))

def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

basket_sets = basket.applymap(encode_units)
print("Shape of one-hot encoded dataset:", basket_sets.shape)

### 3. Generate Frequent Itemsets & Association Rules

In [ ]:
# Generate frequent itemsets using Apriori algorithm with min support threshold
frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)
print("Frequent Itemsets:\n", frequent_itemsets.head())

# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Filter rules using confidence and lift
filtered_rules = rules[(rules['confidence'] >= 0.05) & (rules['lift'] >= 1.0)]
print("\nFiltered Association Rules:\n", filtered_rules.head())

### 4. Visualization of Apriori Results

In [ ]:
# Bar chart of top 10 frequent itemsets
plt.figure(figsize=(10, 5))
top_items = frequent_itemsets.sort_values('support', ascending=False).head(10)
sns.barplot(x=top_items['support'], y=top_items['itemsets'].apply(lambda x: ', '.join(list(x))), palette='viridis')
plt.title('Top 10 Frequent Itemsets')
plt.xlabel('Support')
plt.ylabel('Itemsets')
plt.show()

# Support vs Confidence Plot
plt.figure(figsize=(8, 5))
sns.scatterplot(x='support', y='confidence', size='lift', data=filtered_rules, hue='lift', palette='viridis')
plt.title('Support vs Confidence Plot')
plt.show()

# Network graph of rules
fig, ax = plt.subplots(figsize=(10, 8))
G = nx.DiGraph()
for _, row in filtered_rules.head(20).iterrows():
    ant = ', '.join(list(row['antecedents']))
    con = ', '.join(list(row['consequents']))
    G.add_edge(ant, con, weight=row['lift'])

pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=2000, font_size=10, ax=ax)
plt.title('Network Graph of Association Rules (Top 20)')
plt.show()

---

## SCENARIO 2 – DIMENSIONALITY REDUCTION USING PCA
**Problem Statement:** Reduce high-dimensional data into lower dimensions while preserving maximum variance using PCA.

### 1. Load and Preprocess Data

In [ ]:
# Using the Iris dataset for PCA
iris = load_iris()
df_pca = pd.DataFrame(data=iris.data, columns=iris.feature_names)
y_pca = iris.target

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)
print("Original data shape:", X_scaled.shape)

### 2. Apply PCA

In [ ]:
# Determine components
pca = PCA()
pca.fit(X_scaled)

# Explained variance and cumulative variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("Explained Variance Ratio:\n", explained_variance)
print("Cumulative Variance:\n", cumulative_variance)

### 3. Visualization of PCA Variance

In [ ]:
# Scree plot & Cumulative variance graph
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, align='center', label='Individual explained variance')
plt.step(range(1, len(cumulative_variance) + 1), cumulative_variance, where='mid', label='Cumulative explained variance', color='red')
plt.ylabel('Explained variance ratio')
plt.xlabel('Principal components')
plt.title('Scree Plot and Cumulative Variance')
plt.legend(loc='best')
plt.xticks(range(1, len(explained_variance) + 1))

plt.tight_layout()
plt.show()

### 4. Reduce Dimensions and Visualize transformed data

In [ ]:
# Reduce to 2 dimensions for visualization
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

# 2D Scatter plot
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_pca, cmap='viridis', edgecolor='k', s=60)
plt.title('2D Scatter Plot of Principal Components')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

# Legend
handles, _ = scatter.legend_elements()
plt.legend(handles, iris.target_names, title="Classes")
plt.grid(True)
plt.show()